# TF01 — 1段だけの小さな GPT で和歌（五七五七七）の続きを詠ませる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilu-h-bando/dl-tutorial/blob/main/transformer/TF01_tiny_gpt_waka.ipynb)

このノートブックは **NumPy だけ**（PyTorch などのフレームワークなし）で、GPT 型（デコーダのみ）の
Transformer を **たった1段** だけ組み立て、**八代集（古今和歌集・後撰和歌集・拾遺和歌集・千載和歌集・新古今和歌集）の和歌**を学習させます。
**歌の最初の 5 音（または 5＋7 音）を入力すると、残りを詠んで「五・七・五・七・七」の和歌にします。**

講義の **「Transformer」パート（P21〜39）** の部品を、1つずつ自分の手で動かして確かめます。

| 講義のページ | このノートの節 |
|---|---|
| P24〜25 入力エンコーディング・Dense Vector | 2. トークン化と Embedding |
| P26 ポジション・エンコーディング | 3. 位置情報を足す |
| P27〜28 Self-Attention（Q・K・V） | 4. Self-Attention |
| P30 Masked Attention | 5. 未来を見せない（マスク） |
| P29 Multi-Head Attention | 6. 複数のヘッド |
| P31 Feed Forward（Up・Gate・Down） | 7. MLP（SwiGLU） |
| P34〜35 Residual・LayerNorm | 8. 残差接続と LayerNorm |
| P36〜37 Linear・Softmax | 9. 出力層（Linear → Softmax） |
| P32〜33 Transformer の直列化 | 10. 部品を組み立てる（※1段だけ） |

**このノートで体験すること**
- 学習前は文字がでたらめに並ぶ → 学習すると「はるかすみ／たてるやいつこ／…」のように、**五七五七七の形をした和歌のような並び**が出てくる
- 「区切り記号（／）」もただの 1 文字として学習させるだけで、**音数のリズムまで学ぶ**かを確かめる（音数が合っているかを自動で数える）
- 入力は歌の「最初の部分」。その**続き**を作る（入力文を和歌に*変換*するのではありません）

> 注意：1段・約 7 万パラメータ・約 20 万文字の学習なので、出てくるのは**「和歌らしい形と言葉の並び」**です。
> 意味の通る名歌にはなりません。意味まで理解するのは、層もデータも桁違いに大きい GPT / Llama です（P38〜39 スケーリング則）。

## 動かし方

- **Google Colab**：上の「Open in Colab」バッジ →「ランタイム > すべてのセルを実行」
- **手元の Jupyter**：上から順にセルを実行

追加ライブラリのインストールは不要です（`numpy`・`matplotlib` のみ。Colab には導入済み）。
学習は CPU で数分です。セルは**上から順に**実行してください（インターネットに接続している必要があります）。

## 1. 準備：設定と学習データ

### モデルの大きさ（超小型）
- `CTX = 48`：一度に見られる文字数（コンテキスト長）。区切り記号込みで 1 首が約 36 文字なので、1 首ぶんが丸ごと入ります
- `D = 64`：1文字を表すベクトルの次元（`d_model`）
- `N_HEAD = 4`：アテンションのヘッド数
- `D_FF = 256`：MLP の中間次元（`d` の 4 倍）

### 学習データ：Wikisource の八代集（ひらがな読みつき）
- [Wikisource](https://ja.wikisource.org/) の『古今和歌集』『後撰和歌集』『拾遺和歌集』『千載和歌集』『新古今和歌集』
  （作品は著作権切れ。ページの文章のライセンスは CC BY-SA 4.0）。実行時に Wikisource の API から取得します
- 各歌に、**五・七・五・七・七で区切ったひらがなの読み**が付いています（歴史的仮名遣い）

```
としのうちに春はきにけりひととせをこそとやいはむことしとやいはむ     ← 本文（漢字まじり）
としのうちに－はるはきにけり－ひととせを－こそとやいはむ－ことしとやいはむ   ← 読み（今回使う）
```

### なぜひらがなの「読み」を使うのか
和歌の五七五七七は**音の数（音数）**です。「春」は漢字 1 文字でも 2 音（は・る）なので、音数を数えるにはひらがなにする必要があります。
今回は読みの行だけを使い、**音数がちょうど 5・7・5・7・7 の歌だけ**を学習に使います（字余り・字足らずの歌は除きます）。

### 音数の数え方
ひらがな 1 文字 = 1 音。ただし小さい「ゃ ゅ ょ ぁ ぃ ぅ ぇ ぉ ゎ」は前の文字と合わせて 1 音と数えます。

In [ ]:
import numpy as np
import re, json, time, urllib.request, urllib.parse
import matplotlib.pyplot as plt

# ---- モデルの大きさ（超小型） ----
CTX    = 48      # コンテキスト長（一度に見る文字数）
D      = 64      # 埋め込み次元 d_model
N_HEAD = 4       # ヘッド数
D_FF   = 256     # MLP の中間次元
HD     = D // N_HEAD   # 1ヘッドあたりの次元

# ---- 学習の設定 ----
STEPS        = 3000    # 学習ステップ数（増やすと賢くなるが時間がかかる）
BATCH        = 32      # 1ステップで見る文章の数（ミニバッチ）
LR           = 3e-3    # 学習率
WEIGHT_DECAY = 0.01    # 重み減衰（AdamW）
DROPOUT      = 0.1     # ドロップアウト率（過学習防止）
USE_MASK     = True    # 未来を見せないマスクを使うか（False にして違いを見てみよう）
ALIGN_TO_POEM = True   # 学習データの切り出しを「歌の頭から」にそろえるか（False にして違いを見てみよう）

TARGET = [5, 7, 5, 7, 7]    # 和歌の音数
np.random.seed(0)

In [ ]:
API = "https://ja.wikisource.org/w/api.php"
ANTHOLOGIES = ["古今和歌集", "後撰和歌集", "拾遺和歌集", "千載和歌集", "新古今和歌集"]
HEADERS = {"User-Agent": "tiny-gpt-tutorial/1.0 (ILU deep learning lecture; https://github.com/ilu-h-bando/dl-tutorial)"}

def api_get(**params):
    url = API + "?" + urllib.parse.urlencode({**params, "format": "json", "formatversion": "2"})
    with urllib.request.urlopen(urllib.request.Request(url, headers=HEADERS), timeout=60) as r:
        return json.loads(r.read().decode("utf-8"))

def load_anthology(name):
    """歌集の「巻」ごとのページ（例：古今和歌集/巻一）の本文を取得する"""
    q = api_get(action="query", list="allpages", apprefix=name + "/巻", aplimit=50)
    titles = [p["title"] for p in q["query"]["allpages"]]
    pages = api_get(action="query", prop="revisions", rvprop="content", rvslots="main", titles="|".join(titles))
    return {p["title"]: p["revisions"][0]["slots"]["main"]["content"] for p in pages["query"]["pages"]}

try:
    wikitext = {name: load_anthology(name) for name in ANTHOLOGIES}
except Exception as e:
    raise RuntimeError("Wikisource からの取得に失敗しました。ネットワークを確認して、もう一度このセルを実行してください: " + str(e))

for name, pages in wikitext.items():
    print(f"{name}: {len(pages)} 巻 / {sum(len(t) for t in pages.values()):,} 文字")

In [ ]:
SMALL = set("ぁぃぅぇぉゃゅょゎ")
def morae(s):
    """音数：小さい「ゃゅょ」などは前の文字と合わせて 1 音"""
    return sum(1 for c in s if c not in SMALL)

def parse_poems(wikitext):
    """各歌の「ひらがなの読み」行（－で区切られている）を取り出し、5-7-5-7-7 の歌だけ返す"""
    poems = []
    for body in re.split(r'<span id="\d+">\d+</span>', wikitext)[1:]:       # 1首ずつに分ける
        for line in body.split("\n"):
            line = line.strip()
            if "－" in line and re.fullmatch(r"[ぁ-ゖ－]+", line):          # ひらがなの読みの行
                segs = line.split("－")
                if [morae(s) for s in segs] == TARGET:                       # 音数がちょうど 5・7・5・7・7
                    poems.append(segs)
                break
    return poems

poems, seen = [], set()
for name, pages in wikitext.items():
    n0 = len(poems)
    for title, wt in pages.items():
        for segs in parse_poems(wt):
            if tuple(segs) not in seen:                # 同じ歌の重複は除く
                seen.add(tuple(segs)); poems.append(segs)
    print(f"{name}: 5-7-5-7-7 の歌 {len(poems) - n0} 首")
print("合計:", len(poems), "首")
for segs in poems[:3]:
    print("  ", " / ".join(segs))

## 2. トークン化と Embedding（P24〜25）

### トークナイズ
講義では BPE や SentencePiece（サブワード）を紹介しました。ここでは最小・最速の方法として
**1文字 = 1トークン** にします（文字単位）。文字に番号（トークン ID）を振るだけです。
読みはひらがなだけなので、語彙（文字の種類）は 50 ほどしかありません（『人間失格』は約 1,700 種類でした）。

### 学習用の文字列の作り方
1 首を「句 ／ 句 ／ 句 ／ 句 ／ 句 ＋ 改行」とつなぎます。

```
はるかすみ／たてるやいつこ／みよしのの／よしののやまに／ゆきはふりつつ⏎
```

- **区切り記号 `／`** もふつうの 1 文字（トークン）として扱います。モデルは「5 文字ぐらい進んだら `／` が来る」というリズムを**自分で見つけて**学びます
- **改行** は歌の終わり（と次の歌の始まり）の印です。生成するときは、改行が出たら終わりにします

### Embedding（Dense Vector）
トークン ID は単なる番号なので、そのままでは計算できません。**ID ごとに `D` 次元のベクトル**を1本ずつ持つ表
`E`（語彙数 × `D`）を用意し、ID の行を取り出します。これが Embedding です。
最初は乱数ですが、学習で「意味の近い文字は近いベクトル」になるように更新されます。

In [ ]:
SEP, END = "／", "\n"                       # 句の区切り記号 / 歌の終わり
def poem_text(segs):
    return SEP.join(segs) + END

# 歌を訓練用と検証用に分ける（検証用 = 学習に使わない歌 10%）
order = np.random.default_rng(0).permutation(len(poems))
n_val = len(poems) // 10
val_poems   = [poems[i] for i in order[:n_val]]
train_poems = [poems[i] for i in order[n_val:]]
train_text = "".join(poem_text(p) for p in train_poems)
val_text   = "".join(poem_text(p) for p in val_poems)

chars = sorted(set(train_text + val_text))     # 使われている文字の一覧 = 語彙
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

def encode(s):  return [stoi[c] for c in s if c in stoi]
def decode(ids): return "".join(itos[i] for i in ids)

train = np.array(encode(train_text), dtype=np.int64)
val   = np.array(encode(val_text), dtype=np.int64)
print("語彙数 V =", V, "/ 訓練:", len(train_poems), "首", len(train), "文字 / 検証:", len(val_poems), "首", len(val), "文字")

sample = poem_text(train_poems[0])
ids = encode(sample)
print(sample.replace(END, "⏎"), "→", ids)

# Embedding 表（乱数で初期化）
rng0 = np.random.default_rng(0)
E_demo = rng0.normal(0, 0.02, (V, D))
print("Embedding 表の形:", E_demo.shape, "（語彙数 × D）")
print(f"「{sample[0]}」のベクトル（先頭8次元）:", np.round(E_demo[ids[0], :8], 3))

### 学習データの切り出し方（音数を数えやすくする工夫）

学習では、文字列から `CTX` 文字ずつ切り出して「次の文字を当てる」練習をします。どこから切り出すかで、実は学習のしやすさが大きく変わります。

- `ALIGN_TO_POEM = False`：文字列の**どこからでも**切り出す（歌の途中から始まることが多い）
- `ALIGN_TO_POEM = True`：必ず**歌の頭（改行の位置）から**切り出す

歌を詠ませるときは、いつも「歌の頭」から 1 文字ずつ作り始めます。学習のときも歌の頭から切り出しておけば、
**「いま何文字目か」（P26 の位置エンコーディング）が学習と生成でそろう**ので、「5 音で区切る・7 音で区切る」を数えやすくなります。
（`True` と `False` で、音数が合う歌の割合がどれだけ変わるか、あとで確かめられます）

In [ ]:
def get_batch(d, batch, rng):
    """CTX 文字ずつ切り出す。正解 y は x を 1 文字ずらしたもの"""
    if ALIGN_TO_POEM:
        starts = np.where(d == stoi[END])[0]            # 歌の頭（直前の改行）の位置の一覧
        starts = starts[starts + CTX + 1 < len(d)]
        ix = rng.choice(starts, batch)                   # 必ず歌の頭から切り出す
    else:
        ix = rng.integers(0, len(d) - CTX - 1, batch)    # どこからでも切り出す
    x = np.stack([d[i:i + CTX] for i in ix])
    y = np.stack([d[i + 1:i + CTX + 1] for i in ix])
    return x, y

x, y = get_batch(train, 1, np.random.default_rng(1))
print("入力 x:", decode(x[0]).replace(END, "⏎"))
print("正解 y:", decode(y[0]).replace(END, "⏎"), "  ← 1文字ずれている（次の文字を当てる練習）")

## 3. 位置情報を足す（P26 ポジション・エンコーディング）

Attention は「全単語 ↔ 全単語」の関係を**同時**に計算するので、そのままでは語順がわかりません
（「犬が人を噛んだ」と「人が犬を噛んだ」が区別できない）。
そこで Embedding に「何番目の文字か」を表すベクトルを**足し算**します。

ここでは元論文（Attention Is All You Need）の **sin / cos による位置エンコーディング**を使います。
位置ごとに異なる波のパターンになるので、位置が識別できます（RoPE は最近の LLM の方式で、考え方は同じ「位置を表す」です）。

In [ ]:
def pos_encoding(n, d):
    pos = np.arange(n)[:, None]
    i = np.arange(d // 2)[None, :]
    ang = pos / (10000 ** (2 * i / d))
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(ang)
    pe[:, 1::2] = np.cos(ang)
    return pe

PE = pos_encoding(CTX, D)

plt.figure(figsize=(8, 3))
plt.imshow(PE, aspect="auto", cmap="RdBu")
plt.xlabel("dimension"); plt.ylabel("position"); plt.title("Positional Encoding (sin / cos)")
plt.colorbar(); plt.show()

# 入力ベクトル = Embedding + 位置エンコーディング
x0 = E_demo[ids] + PE[:len(ids)]
print("Transformer への入力の形:", x0.shape, "（文字数 × D）")

## 4. Self-Attention（P27〜28）

各文字のベクトル `x` から、3 種類のベクトルを作ります。

- **Q (Query)**：`Q = x Wq` … 「私は今どんな情報を探している？」
- **K (Key)**：`K = x Wk` … 「私はこんな情報を持っています」
- **V (Value)**：`V = x Wv` … 「選ばれたら、この中身を渡します」

関連度（スコア）は `Q × Kᵀ`、それを **softmax** で「注目度（合計 1）」に直し、その重みで `V` を足し合わせます。

`Attention(Q,K,V) = softmax(Q Kᵀ / √d) V`

`√d` で割るのは、次元が大きいとスコアが大きくなりすぎて softmax が極端になるのを防ぐためです。
下は、まだ**学習していない乱数の重み**での計算例です（注目先に意味はありません。「計算の流れ」を見てください）。

In [ ]:
def softmax(x):
    e = np.exp(x - x.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)

def show_matrix(M, labels, title):
    print(title)
    print("      " + "  ".join(f"{c:>4s}" for c in labels))
    for lab, row in zip(labels, M):
        print(f"  {lab:>2s}  " + "  ".join(f"{v:4.2f}" for v in row))

s = "はるかすみ／たてるや"
xs = E_demo[encode(s)] + PE[:len(s)]            # (文字数, D)
r = np.random.default_rng(1)
Wq, Wk, Wv = (r.normal(0, 0.3, (D, D)) for _ in range(3))

Q, K, Vv = xs @ Wq, xs @ Wk, xs @ Wv
scores = Q @ K.T / np.sqrt(D)                   # 関連度スコア（文字数 × 文字数）
A = softmax(scores)                              # 注目度（各行の合計 = 1）
out = A @ Vv                                     # V の重み付き和

show_matrix(A, list(s), "注目度 A（行 = 見ている文字、列 = 見られる文字。各行の合計は 1）")
print("\n各行の合計:", np.round(A.sum(-1), 3))
print("出力の形:", out.shape, "（入力と同じ形。各文字のベクトルが、文脈を取り込んだものに置き換わる）")

## 5. 未来を見せない（P30 Masked Attention）

GPT は「次の文字を予測する」モデルです。学習中に**答え（右側＝未来の文字）が見えてしまうとズル**になります。
そこで、スコアの「未来」の部分を **−∞** にしてから softmax します。`exp(−∞) = 0` なので、注目度が 0 になります。
行列で見ると、**右上の三角形が見えなくなる**形です。

In [ ]:
L = len(s)
mask = np.triu(np.ones((L, L), dtype=bool), 1)       # 右上 = 未来 = True
masked_scores = np.where(mask, -1e9, scores)         # 未来のスコアを −∞（に近い値）にする
A_masked = softmax(masked_scores)

show_matrix(A_masked, list(s), "マスクあり：右上（未来）の注目度が 0 になる")

## 6. 複数のヘッド（P29 Multi-Head Attention）

1 種類の注目の仕方だけでは、1 種類の関係しか捉えられません。
そこで `D` 次元を `N_HEAD` 個に**分割**し、ヘッドごとに別々の `Wq, Wk, Wv` で Attention を同時に計算します。
（例：あるヘッドは直前の文字、別のヘッドは文の主語…を見る、のように役割分担することが期待されます）
最後に各ヘッドの出力をつなげて、`Wo` で元の次元に混ぜ直します。

ここまでを**関数**にまとめます（学習で使うので、バッチ対応・逆伝播用の途中結果の保存つきです）。

In [ ]:
def attn_f(x, P, causal=True):
    """マスク付き Multi-Head Self-Attention。x: (B, L, D) → (B, L, D)"""
    B, L, _ = x.shape
    dt = x.dtype.type
    split = lambda t: t.reshape(B, L, N_HEAD, HD).transpose(0, 2, 1, 3)   # (B, ヘッド, L, HD)
    q = split(x @ P["Wq"])
    k = split(x @ P["Wk"])
    v = split(x @ P["Wv"])
    sc = q @ k.transpose(0, 1, 3, 2) * dt(1 / np.sqrt(HD))                 # (B, ヘッド, L, L)
    if causal:
        sc = np.where(np.triu(np.ones((L, L), dtype=bool), 1), dt(-1e9), sc)
    A = softmax(sc)
    o = (A @ v).transpose(0, 2, 1, 3).reshape(B, L, D)                     # ヘッドをつなげる
    return o @ P["Wo"], (x, q, k, v, A, o)

# 動作確認（乱数の重み）
Pd = {n: r.normal(0, 0.2, (D, D)) for n in ("Wq", "Wk", "Wv", "Wo")}
out, c = attn_f(xs[None], Pd, causal=USE_MASK)
print("入力:", xs[None].shape, "→ 出力:", out.shape, " / 注目度 A の形:", c[4].shape, "（バッチ, ヘッド数, L, L）")

## 7. MLP（P31 Feed Forward：Up・Gate・Down）

Attention が文脈を**混ぜる**役なら、MLP は混ぜた結果を**文字ごとに加工する**役です。
「広げる → 選ぶ → 圧縮する」の 3 ステップです。

1. **Up**：`D` 次元 → `D_FF` 次元へ広げる（`x Wu`）
2. **Gate**：別の広げ方（`x Wg`）に活性化関数（SiLU）をかけて「どの情報を通すか」のスイッチを作る
3. **Down**：ゲートを通った情報を `D` 次元に戻す（`Wd`）

`MLP(x) = ( SiLU(x Wg) ⊙ (x Wu) ) Wd`（⊙ は要素ごとの掛け算）…これが **SwiGLU**（Llama 系で使われる形）です。

パラメータ数は 3 × `D` × `D_FF` で、Attention（4 × `D` × `D`）より多く、**モデルの知識の多くは MLP に入る**といわれます。

In [ ]:
def ffn_f(x, P):
    """SwiGLU 型 MLP。x: (B, L, D) → (B, L, D)"""
    a = x @ P["Wg"]                 # Gate 用
    b = x @ P["Wu"]                 # Up
    sg = 1 / (1 + np.exp(-a))
    s_ = a * sg                     # SiLU(a) = a × sigmoid(a)
    h = s_ * b                      # ゲートを通す（要素ごとの積）
    return h @ P["Wd"], (x, a, b, sg, s_, h)   # Down

print("Attention のパラメータ数: 4 × D × D      =", 4 * D * D)
print("MLP のパラメータ数      : 3 × D × D_FF   =", 3 * D * D_FF)

## 8. 残差接続と LayerNorm（P34〜35）

### LayerNorm
文字ごとのベクトルの値のバラつき（平均と分散）を整えて、学習を安定させます。
`(x − 平均) / 標準偏差` にしたあと、学習できるスケール `g` とずれ `b` を掛け足しします。

### 残差接続（スキップ接続）
`x → x + 変換(x)` のように、**元の入力をそのまま足して**次へ渡します。
「何もしない」が簡単にでき、勾配も流れやすいので、層を深くできます（講義の勾配消失の話の解決策のひとつです）。

並べ方は **Pre-LN**（先に LayerNorm、それから Attention / MLP、最後に残差を足す）にします。現在の LLM の主流です。

```
x → x + Attention( LayerNorm(x) )
  → x + MLP( LayerNorm(x) )
```

In [ ]:
def ln_f(x, g, b):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    inv = 1 / np.sqrt(var + 1e-5)
    xh = (x - mu) * inv
    return xh * g + b, (xh, inv, g)

z = np.random.default_rng(2).normal(5, 3, (1, 4, D))          # 平均 5・バラつき 3 のベクトル
zn, _ = ln_f(z, np.ones(D), np.zeros(D))
print("LayerNorm 前: 平均 %.2f / 標準偏差 %.2f" % (z.mean(), z.std()))
print("LayerNorm 後: 平均 %.2f / 標準偏差 %.2f   ← 整った" % (zn.mean(), zn.std()))

## 9. 出力層（P36〜37 Linear → Softmax）

最後に、各位置のベクトルを **Linear（`D` → 語彙数 `V`）** で「次の文字の候補すべての得点（logits）」にして、
**Softmax** で確率（合計 1）にします。正解の文字の確率が高くなるように学習します。

損失は **クロスエントロピー**：正解の文字に付けた確率を `p` として `−log p`（小さいほどよい）。
学習前は全文字をほぼ同じ確率で選ぶので、損失は `ln(V)` に近い値になります。

In [ ]:
print("ln(V) =", round(np.log(V), 3), "  ← 学習前の損失はこの値に近くなるはず")

logits = np.random.default_rng(3).normal(0, 0.1, V)
p = softmax(logits)
print("確率の合計:", round(p.sum(), 3), "/ 最大の確率:", round(p.max(), 5), "（ほぼ均等）")

## 10. 部品を組み立てる（P32〜33 Transformer の直列化）

ここまでの部品を並べると、GPT の **1 ブロック**になります。

```
文字ID → Embedding + 位置エンコーディング
       → [ LayerNorm → Attention → +残差 → LayerNorm → MLP → +残差 ] × N 層   ← 今回は N = 1
       → LayerNorm → Linear → Softmax → 次の文字の確率
```

講義の P33 のように、このブロックを縦に何層も積むのが本物の GPT です。今回はあえて **1 層だけ**にしています
（層を増やす = `forward` の中のブロックを繰り返すだけです。「やってみよう」参照）。

In [ ]:
def init_params(seed=0, dtype=np.float32):
    r = np.random.default_rng(seed)
    n = lambda *shape: r.normal(0, 0.02, shape)
    P = {
        "E": n(V, D),                                                  # Embedding 表
        "ln1_g": np.ones(D), "ln1_b": np.zeros(D),
        "Wq": n(D, D), "Wk": n(D, D), "Wv": n(D, D), "Wo": n(D, D),   # Attention
        "ln2_g": np.ones(D), "ln2_b": np.zeros(D),
        "Wg": n(D, D_FF), "Wu": n(D, D_FF), "Wd": n(D_FF, D),         # MLP (Gate, Up, Down)
        "lnf_g": np.ones(D), "lnf_b": np.zeros(D),
        "Wout": n(D, V),                                               # 出力層
    }
    return {k: v.astype(dtype) for k, v in P.items()}

def make_dropout(shape, rate, rng, dtype):
    if rate == 0 or rng is None:
        return None
    return (rng.random(shape) >= rate).astype(dtype) / dtype(1 - rate)

def forward(P, x_ids, y_ids=None, rng=None, dropout=0.0):
    """x_ids: (B, L) の文字ID。y_ids を渡すと損失も計算する"""
    B, L = x_ids.shape
    dt = P["E"].dtype.type
    m0, m1, m2 = (make_dropout((B, L, D), dropout, rng, dt) for _ in range(3))

    h0 = P["E"][x_ids] + pos_encoding(L, D).astype(dt)          # Embedding + 位置
    if m0 is not None: h0 = h0 * m0

    n1, c_ln1 = ln_f(h0, P["ln1_g"], P["ln1_b"])                # ── ブロック（1層）──
    ao, c_att = attn_f(n1, P, causal=USE_MASK)                  #   Attention
    if m1 is not None: ao = ao * m1
    h1 = h0 + ao                                                #   + 残差

    n2, c_ln2 = ln_f(h1, P["ln2_g"], P["ln2_b"])
    mo, c_ffn = ffn_f(n2, P)                                    #   MLP
    if m2 is not None: mo = mo * m2
    h2 = h1 + mo                                                #   + 残差

    nf, c_lnf = ln_f(h2, P["lnf_g"], P["lnf_b"])                # 最終 LayerNorm
    logits = nf @ P["Wout"]                                     # Linear → 各文字の得点
    p = softmax(logits)                                         # Softmax → 確率
    cache = dict(x_ids=x_ids, y_ids=y_ids, masks=(m0, m1, m2), n1=n1, c_ln1=c_ln1, c_att=c_att,
                 n2=n2, c_ln2=c_ln2, c_ffn=c_ffn, nf=nf, c_lnf=c_lnf, p=p)
    if y_ids is None:
        return logits, cache
    loss = -np.log(p.reshape(-1, V)[np.arange(B * L), y_ids.reshape(-1)] + 1e-12).mean()
    return loss, cache

P = init_params()
print("パラメータ数:", sum(v.size for v in P.values()))
for k, v in P.items():
    print(f"  {k:6s} {str(v.shape):12s} {v.size:7d}")

loss0, _ = forward(P, *get_batch(train, 32, np.random.default_rng(0)))
print("\n学習前の損失:", round(float(loss0), 3), " ≒ ln(V) =", round(np.log(V), 3))

### 歌を詠むための部品（生成）

文字を 1 つ選ぶたびに、**確率に従ってサイコロを振る**のが生成です。

- `temperature`（温度）：低いと最有力候補ばかり選ぶ（安定）、高いとばらける（多様）
- `top_k`：確率の高い上位 k 文字だけを候補にする（暴走を防ぐ）

和歌用に、次の部品を用意します。

- `format_prompt`：入力した最初の部分（ひらがな）を、5・7 音ごとに区切り記号 `／` を入れた形に整える
  （「はるかすみ」→「はるかすみ／」、「はるかすみたてるやいつこ」→「はるかすみ／たてるやいつこ／」。区切りや空白を入れても無視します）
- `generate_waka`：整えた書き出しの続きを 1 文字ずつ作る。**改行が出たら終わり**
- `is_waka`：できた歌が 5・7・5・7・7 の音数になっているかを数えて判定する
- `force_rhythm=True`：音数を**強制**するオプション。モデルが選んだ文字はそのままで、`／` や改行を出してよいタイミングだけをルールで決める
  （`False` なら音数もモデルまかせ。モデルが音数のリズムを学べたかを見るために使います）

学習前はどの文字も同じ確率なので、でたらめな文字列になります。

In [ ]:
import unicodedata
def normalize_kana(s):
    """カタカナ → ひらがな、濁点・半濁点を外す（昔の表記には濁点がないため。例：「かぜ」→「かせ」）"""
    s = "".join(chr(ord(c) - 0x60) if "ァ" <= c <= "ヶ" else c for c in s)
    return "".join(c for c in unicodedata.normalize("NFD", s) if c not in "\u3099\u309a")

def format_prompt(kana):
    """入力（ひらがな）を、5・7 音ごとに区切り記号を入れた書き出しにする"""
    chars = [c for c in kana if c not in "／－/ 　\n"]
    segs, cur = [], ""
    for c in chars:
        if c not in SMALL and len(segs) < 4 and cur and morae(cur) >= TARGET[len(segs)]:
            segs.append(cur); cur = ""          # 音数がそろったので句を区切る
        cur += c
    if len(segs) < 4 and cur and morae(cur) == TARGET[len(segs)]:
        segs.append(cur); cur = ""              # ちょうどで終わったら、次の句から始める
    body = SEP.join(segs + ([cur] if cur else []))
    return body + SEP if (not cur and segs and len(segs) < 5) else body

def is_waka(text):
    segs = text.strip(END).split(SEP)
    return len(segs) == 5 and [morae(s) for s in segs] == TARGET

def show(text):
    mark = "✓" if is_waka(text) else "✗"
    return f"{text}   {mark} 音数 " + "-".join(str(morae(s)) for s in text.split(SEP))

def generate_waka(P, kana, temperature=0.7, top_k=8, force_rhythm=False, seed=None, max_len=60):
    """歌の最初の部分（ひらがな）から、残りを 1 文字ずつ作って歌全体を返す"""
    r = np.random.default_rng(seed)
    kana = normalize_kana(kana)
    unknown = [c for c in kana if c not in stoi and c not in "／－/ 　"]
    if unknown:
        print("※学習データにない文字は無視します:", "".join(unknown))
    prompt = format_prompt([c for c in kana if c in stoi or c in "／－/ 　"])
    ids = encode(END + prompt)                           # 先頭に改行（= 歌の始まり）を置く
    sep, end = stoi[SEP], stoi[END]
    for _ in range(max_len):
        logits, _ = forward(P, np.array(ids[-CTX:])[None])
        lg = logits[0, -1] / max(temperature, 1e-6)
        if force_rhythm:                                  # 音数のルールで、出してよい文字を絞る
            segs = decode(ids[1:]).split(SEP)
            k, m = len(segs) - 1, morae(segs[-1])         # k = 完成した句の数, m = いまの句の音数
            ok = np.ones(V, dtype=bool)
            if k < 5 and m < TARGET[k]: ok[[sep, end]] = False     # まだ区切れない
            else:
                ok[:] = False
                ok[sep if k < 4 else end] = True                    # ちょうど → 区切る（最後は終わる）
            lg = np.where(ok, lg, -1e9)
        if top_k:
            th = np.sort(lg)[-top_k]
            lg = np.where(lg < th, -1e9, lg)
        nxt = int(r.choice(V, p=softmax(lg)))
        if nxt == end:
            break
        ids.append(nxt)
    return decode(ids[1:])

print("「はるかすみたてるやいつこ」→", format_prompt("はるかすみたてるやいつこ"))
print(show(generate_waka(P, "はるかすみ", seed=0)))

## 11. 学習（逆伝播と AdamW）

学習の 1 ステップは、講義で学んだ流れそのものです。

1. **順伝播**：文字列を入れて、次の文字の確率を出す
2. **損失**：正解の文字の確率から損失を計算
3. **逆伝播**：損失を、出力側から入力側へ連鎖律で伝えて、全パラメータの勾配を求める
4. **更新**：勾配降下法（ここでは改良版の **AdamW**）で重みを少し動かす

逆伝播のコードは長いので、**次のセルは読み飛ばして構いません**（P13〜14 の連鎖律を各部品で書いているだけです）。
その代わり、その次のセルで「**数値微分と一致するか**」を確かめて、正しさを保証します。

In [ ]:
# ===== 逆伝播（読み飛ばし OK） =====
def ln_b(dy, c):
    xh, inv, g = c
    dg = (dy * xh).sum((0, 1)); db = dy.sum((0, 1)); dxh = dy * g
    dx = inv * (dxh - dxh.mean(-1, keepdims=True) - xh * (dxh * xh).mean(-1, keepdims=True))
    return dx, dg, db

def lin_b(dy, x, W):
    """y = x W の逆伝播。dx と dW を返す"""
    x2 = x.reshape(-1, x.shape[-1]); d2 = dy.reshape(-1, dy.shape[-1])
    return (d2 @ W.T).reshape(x.shape), x2.T @ d2

def attn_b(dout, P, c):
    x, q, k, v, A, o = c
    B, L, _ = x.shape
    dt = x.dtype.type
    g = {}
    do, g["Wo"] = lin_b(dout, o, P["Wo"])
    do = do.reshape(B, L, N_HEAD, HD).transpose(0, 2, 1, 3)
    dA = do @ v.transpose(0, 1, 3, 2)
    dv = A.transpose(0, 1, 3, 2) @ do
    dsc = A * (dA - (dA * A).sum(-1, keepdims=True)) * dt(1 / np.sqrt(HD))   # softmax の逆伝播
    dq = dsc @ k
    dk = dsc.transpose(0, 1, 3, 2) @ q
    merge = lambda t: t.transpose(0, 2, 1, 3).reshape(B, L, D)
    dxq, g["Wq"] = lin_b(merge(dq), x, P["Wq"])
    dxk, g["Wk"] = lin_b(merge(dk), x, P["Wk"])
    dxv, g["Wv"] = lin_b(merge(dv), x, P["Wv"])
    return dxq + dxk + dxv, g

def ffn_b(dout, P, c):
    x, a, b, sg, s_, h = c
    g = {}
    dh, g["Wd"] = lin_b(dout, h, P["Wd"])
    ds, db = dh * b, dh * s_
    da = ds * (sg * (1 + a * (1 - sg)))                                       # SiLU の微分
    dxa, g["Wg"] = lin_b(da, x, P["Wg"])
    dxb, g["Wu"] = lin_b(db, x, P["Wu"])
    return dxa + dxb, g

def backward(P, cache):
    x_ids, y_ids, (m0, m1, m2) = cache["x_ids"], cache["y_ids"], cache["masks"]
    B, L = x_ids.shape
    G = {}
    # 損失 → logits（softmax + クロスエントロピーの勾配は p − 正解）
    dl = cache["p"].copy().reshape(-1, V)
    dl[np.arange(B * L), y_ids.reshape(-1)] -= 1
    dl = (dl / (B * L)).reshape(B, L, V)
    dnf, G["Wout"] = lin_b(dl, cache["nf"], P["Wout"])
    dh2, G["lnf_g"], G["lnf_b"] = ln_b(dnf, cache["c_lnf"])
    # MLP ブロック
    dmo = dh2 * m2 if m2 is not None else dh2
    dn2, gf = ffn_b(dmo, P, cache["c_ffn"]); G.update(gf)
    dh1_ln, G["ln2_g"], G["ln2_b"] = ln_b(dn2, cache["c_ln2"])
    dh1 = dh2 + dh1_ln                                  # 残差：勾配はそのまま通り抜ける
    # Attention ブロック
    dao = dh1 * m1 if m1 is not None else dh1
    dn1, ga = attn_b(dao, P, cache["c_att"]); G.update(ga)
    dh0_ln, G["ln1_g"], G["ln1_b"] = ln_b(dn1, cache["c_ln1"])
    dh0 = dh1 + dh0_ln
    if m0 is not None: dh0 = dh0 * m0
    G["E"] = np.zeros_like(P["E"])
    np.add.at(G["E"], x_ids, dh0)                       # Embedding 表の該当行に勾配を足す
    return G

In [ ]:
# ===== 勾配チェック：逆伝播の勾配 vs 数値微分（f(w+ε) − f(w−ε)）/ 2ε =====
Pc = {k: v.astype(np.float64) for k, v in init_params(seed=1).items()}
xb, yb = get_batch(train, 4, np.random.default_rng(1))
loss, cache = forward(Pc, xb, yb)
Gc = backward(Pc, cache)
rc = np.random.default_rng(5)
print(f"{'param':7s} {'逆伝播':>12s} {'数値微分':>12s}   相対誤差")
worst = 0
for name in Pc:
    if name == "E":
        idx = (int(xb[0, 0]), 3)             # 実際に使われた文字の行を調べる
    else:
        idx = tuple(int(rc.integers(0, s)) for s in Pc[name].shape)
    eps, old = 1e-5, Pc[name][idx]
    Pc[name][idx] = old + eps; lp, _ = forward(Pc, xb, yb)
    Pc[name][idx] = old - eps; lm, _ = forward(Pc, xb, yb)
    Pc[name][idx] = old
    num, ana = (lp - lm) / (2 * eps), Gc[name][idx]
    rel = abs(ana - num) / (abs(ana) + abs(num) + 1e-12)
    worst = max(worst, rel)
    print(f"{name:7s} {ana:12.4e} {num:12.4e}   {rel:.1e}")
print("\n最大の相対誤差:", f"{worst:.1e}", " → ", "一致 ✓" if worst < 1e-3 else "不一致 ✗")

### 学習ループ

**AdamW**：勾配降下法（講義 P16〜17）の改良版です。
- 過去の勾配の移動平均（慣性）と、勾配の大きさに応じた歩幅の自動調整を使う（Adam）
- 重みが大きくなりすぎないようにブレーキをかける（weight decay）＝過学習防止

その他の工夫：学習の最初は学習率を小さくして徐々に上げ（ウォームアップ）、その後ゆっくり下げます。
勾配が大きすぎるときは切り詰めます（勾配クリッピング）。

学習中は 250 ステップごとに、
- `loss`：訓練データでの損失 / `val`：**学習に使っていない**部分での損失
- 同じ書き出し「はるかすみ」からの生成例（✓ = 音数が 5・7・5・7・7 になった）

を表示します。生成例が少しずつ和歌らしくなっていく（音数がそろってくる）様子を見てください。
（`loss` はドロップアウトでわざと邪魔をした状態で測るため、`val` より高く出ることがあります）

In [ ]:
P = init_params(seed=0)
m = {k: np.zeros_like(v) for k, v in P.items()}      # Adam の 1 次モーメント
v2 = {k: np.zeros_like(v) for k, v in P.items()}     # Adam の 2 次モーメント
rng = np.random.default_rng(0)
hist = {"step": [], "loss": [], "val": []}
run_loss, t0 = [], time.time()

def eval_val(P):
    r = np.random.default_rng(123)
    return float(np.mean([forward(P, *get_batch(val, 32, r))[0] for _ in range(4)]))

for step in range(1, STEPS + 1):
    # 学習率スケジュール：ウォームアップ → コサインで減衰
    lr = LR * min(1, step / 100) * (0.1 + 0.9 * 0.5 * (1 + np.cos(np.pi * step / STEPS)))

    xb, yb = get_batch(train, BATCH, rng)
    loss, cache = forward(P, xb, yb, rng=rng, dropout=DROPOUT)     # ① 順伝播 + ② 損失
    G = backward(P, cache)                                          # ③ 逆伝播
    run_loss.append(float(loss))

    gn = np.sqrt(sum((g ** 2).sum() for g in G.values()))           # 勾配クリッピング
    scale = min(1.0, 1.0 / (gn + 1e-6))
    for k in P:                                                     # ④ AdamW で更新
        g = G[k] * scale
        m[k] = 0.9 * m[k] + 0.1 * g
        v2[k] = 0.999 * v2[k] + 0.001 * g * g
        mh, vh = m[k] / (1 - 0.9 ** step), v2[k] / (1 - 0.999 ** step)
        if P[k].ndim == 2 and k != "E":
            P[k] -= lr * WEIGHT_DECAY * P[k]                        # weight decay
        P[k] -= lr * mh / (np.sqrt(vh) + 1e-8)

    if step % 250 == 0 or step == 1:
        va = eval_val(P)
        hist["step"].append(step); hist["loss"].append(float(np.mean(run_loss))); hist["val"].append(va)
        print(f"step {step:5d}  loss {np.mean(run_loss):.3f}  val {va:.3f}  ({time.time() - t0:.0f}秒)")
        print("    ", show(generate_waka(P, "はるかすみ", seed=0)))
        run_loss = []
print("学習おわり")

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.plot(hist["step"], hist["loss"], "o-", label="train loss")
plt.plot(hist["step"], hist["val"], "s-", label="val loss")
plt.axhline(np.log(V), color="gray", ls=":", label="ln(V) (random guess)")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

## 12. 最初の 5 音から続きを詠ませる

学習に使っていない歌（検証用）の**最初の 5 音**、または**最初の 5＋7 音**だけを与えて、残りを詠ませてみましょう。
原歌と見比べてください（✓ は音数が 5・7・5・7・7 になっている印）。

- 原歌とそっくり（丸写し）になっていれば**暗記**、別の歌になっていれば**新しく詠んだ**ことになります
- 乱数の種（`seed`）を変えると、同じ書き出しでも別の歌になります

In [ ]:
for i, segs in enumerate(val_poems[:5]):
    print("原歌       :", SEP.join(segs))
    print("5音から    :", show(generate_waka(P, segs[0], seed=i)))
    print("5+7音から  :", show(generate_waka(P, segs[0] + segs[1], seed=i)))
    print()

In [ ]:
# 自分で書き出しを決めてみる（歴史的仮名遣いのひらがな）
for i, kana in enumerate(["はるかすみ", "あきのよの", "こひしくは", "ゆきふれは"]):
    for seed in range(2):
        print(f"【{kana}】", show(generate_waka(P, kana, seed=10 * i + seed)))
    print()

### 音数のリズムは学べたか？

学習した結果だけで、5・7・5・7・7 の歌がどれだけ作れるかを数えてみます。
検証用の歌の最初の 5 音を 100 通り与えて、

- **モデルまかせ**：音数が合った歌の割合（区切り記号 `／` を出すタイミングをモデルが決める）
- **丸写し**：できた歌が、**学習に使った歌と完全に同じ**だった割合（暗記していないか）
- **音数を強制**：`force_rhythm=True` にしたとき（音数は必ず合う。丸写しの割合を見る）

を表示します（1〜2 分かかることがあります）。

In [ ]:
train_set = {poem_text(p) for p in train_poems}

def evaluate(P, n=100, force=False):
    ok = copy = 0
    for i, segs in enumerate(val_poems[:n]):
        text = generate_waka(P, segs[0], force_rhythm=force, seed=i)
        ok += is_waka(text)
        copy += (text + END) in train_set
    return ok / n * 100, copy / n * 100

n_eval = min(100, len(val_poems))
free_ok, free_copy = evaluate(P, n_eval, force=False)
forced_ok, forced_copy = evaluate(P, n_eval, force=True)
print(f"モデルまかせ : 音数が合った歌 {free_ok:.0f}%   / 学習データの丸写し {free_copy:.0f}%")
print(f"音数を強制   : 音数が合った歌 {forced_ok:.0f}%   / 学習データの丸写し {forced_copy:.0f}%")

## 自由に入力して試す（入力欄）

下のセルを実行すると**入力欄**が出ます。歌の**最初の部分**（5 音、または 5＋7 音、5＋7＋5 音…）を**ひらがな（歴史的仮名遣い）**で入力して Enter を押すと、
続きを詠んで歌にします。何度でも続けて入力できます。**何も入力せずに Enter を押すと終了**します。

- 「はるかすみ」でも「はるかすみ／たてるやいつこ」でも「はるかすみたてるやいつこ」でも OK（5・7 音ごとの区切りは自動で入ります）
- 学習データの和歌は**昔の表記**です。**濁点・半濁点は書かれていません**（「かぜ」→「かせ」）。入力した濁点は自動で外し、カタカナはひらがなに直します
- 仮名遣いも昔のものです。現代の発音どおりに書くと、学習データにない並びになることがあります（例：「きょう」→「けふ」、「おもう」→「おもふ」）
- 学習データにない文字（漢字 など）は無視されます
- `FORCE_RHYTHM = True` にすると音数を必ず 5・7・5・7・7 にそろえます

In [ ]:
# ===== 入力欄：歌の最初の部分を入れると続きを詠む（空のまま Enter で終了） =====
TEMPERATURE  = 0.7     # 低い = 安定、高い = 多様
TOP_K        = 8       # 確率の高い上位 k 文字から選ぶ（0 で制限なし）
FORCE_RHYTHM = False   # True にすると音数を必ず 5・7・5・7・7 にそろえる
N_CANDIDATES = 5       # 1回の入力につき何首詠むか

while True:
    kana = input("歌の最初の部分（ひらがな。空のまま Enter で終了）> ")
    if not kana.strip():
        print("終了しました")
        break
    kana = normalize_kana(kana)
    if not encode(kana):
        print("  ※学習データにある文字が1つもありません。別の文字で試してください\n")
        continue
    print("  書き出し →", format_prompt(kana))
    for k in range(N_CANDIDATES):
        print(f"  [{k + 1}]", show(generate_waka(P, kana, TEMPERATURE, TOP_K, FORCE_RHYTHM)))
    print()

## 13. Attention を見てみる

学習した 4 つのヘッドが、歌のどこに注目しているかを見てみましょう。
行が「いま見ている文字」、列が「注目される文字」です。右上が真っ白なのは、マスクで未来が見えないためです。
区切り記号 `／` に強く注目しているヘッドが見つかるかもしれません（今いる句が何音目かを数える手がかりになります）。

In [ ]:
# 日本語フォントの設定（Colab では Noto Sans CJK を入れる）
import matplotlib.font_manager as fm
def setup_japanese_font():
    cands = ["Noto Sans CJK JP", "IPAexGothic", "IPAGothic", "Meiryo", "Yu Gothic", "MS Gothic", "Hiragino Sans", "TakaoGothic"]
    def find():
        names = {f.name for f in fm.fontManager.ttflist}
        return next((c for c in cands if c in names), None)
    name = find()
    if name is None:
        try:
            import subprocess
            subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], check=True, capture_output=True)
            for f in fm.findSystemFonts():
                if "NotoSansCJK" in f or "Noto" in f and "CJK" in f:
                    fm.fontManager.addfont(f)
            name = find()
        except Exception as e:
            print("日本語フォントを用意できませんでした（文字が□になる場合があります）:", e)
    if name:
        plt.rcParams["font.family"] = name
    return name

print("使用フォント:", setup_japanese_font())

sent = poem_text(val_poems[0]).replace(END, "")
_, cache = forward(P, np.array([encode(sent)]))
A = cache["c_att"][4][0]                      # (ヘッド数, L, L)
fig, axes = plt.subplots(1, N_HEAD, figsize=(4 * N_HEAD, 4.2))
for h, ax in enumerate(axes):
    ax.imshow(A[h], cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(sent))); ax.set_xticklabels(list(sent), fontsize=7)
    ax.set_yticks(range(len(sent))); ax.set_yticklabels(list(sent), fontsize=7)
    ax.set_title(f"Head {h + 1}")
plt.tight_layout(); plt.show()

## やってみよう

セルの中の設定を変えて、**1. 準備のセルから**（または「すべてのセルを実行」で）もう一度動かしてみましょう。

| 変えるもの | 試すこと | 観察すること |
|---|---|---|
| 入力欄の `FORCE_RHYTHM = True` | 音数を強制 | 音数は必ず 5・7・5・7・7 になる。それでも歌として自然かどうかはモデル（学習の質）しだい |
| `ALIGN_TO_POEM = False` | 切り出しを歌の頭にそろえない | 「音数が合った歌」の割合が下がる（位置エンコーディングが効いていることの確認） |
| 入力の長さ | 5 音 / 5＋7 音 / 5＋7＋5 音 を与える | 与える部分が長いほど、残りの選択肢は少なくなる。原歌に似た歌になるか |
| `TEMPERATURE` / `TOP_K` | 0.3 や 1.2 に変える | 低いと似た言い回しの繰り返し、高いと崩れやすい（✗ が増える） |
| `DROPOUT = 0.0` と `STEPS = 6000` | ドロップアウトなしで長く学習 | 検証 loss が上がり始めないか。「丸写し」の割合が増えないか（暗記） |
| `USE_MASK = False` | マスクを外す | 学習時の loss がどうなるか。生成はうまくいくか（未来が見えると何が起きる？） |
| `ANTHOLOGIES` | 歌集を 1〜2 集に減らす | データが少ないと、音数のリズムや言葉の並びはどこまで学べるか（スケーリング則 P39） |
| `D = 128` / `D_FF = 512` | モデルを大きくする | 表現力が上がる分遅くなる。`val` はどうなる？ |

### 発展：層を増やす（P33）
`forward` の「ブロック（1層）」の部分を `for` 文で繰り返し、層ごとに別のパラメータ（`Wq_1, Wq_2 …`）を持てば、
講義で見た「Attention + MLP を直列に何層も積む」構造になります。逆伝播も同じ部品（`attn_b`, `ffn_b`, `ln_b`）を層の数だけ逆順に呼ぶだけです。

### 発展：漢字まじりの入力・意味の通る歌
このモデルは**ひらがなの読み**しか知りません。「春霞たてる…」のような漢字まじりを入力するには、読みに変換する処理（形態素解析など）が必要です。
また「季節」や「題」を指定して詠ませる、**意味の通る**歌にするには、層・データ・パラメータを桁違いに増やすか、
（題, 歌）のペアでの追加学習（SFT、講義 P41〜）が必要です。

## 出典・ライセンス

- 学習データ：[Wikisource](https://ja.wikisource.org/)（『古今和歌集』『後撰和歌集』『拾遺和歌集』『千載和歌集』『新古今和歌集』）。
  作品は著作権が切れています。ページの文章のライセンスは [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/deed.ja)（Wikisource の寄稿者による）です。
  このノートブックは実行時に取得するだけで、データを再配布していません。
- 参考：Vaswani et al., *Attention Is All You Need* (2017)